# Using Gambit with SageMath

This tutorial demonstrates how to use the Gambit and SageMath Python packages together for game-theoretic analysis.

**SageMath** is a powerful open-source mathematics platform that includes a game theory module with a catalog of classic games, exact symbolic computation, and built-in Nash equilibrium solvers.

**Gambit (pygambit)** is the leading open-source package for game theory computation. It provides a wide range of Nash equilibrium algorithms, supports N-player games, and can return exact rational results.

**Why use both together?**
- SageMath has a ready-to-use catalog of classic games and exact rational arithmetic
- Gambit provides more equilibrium algorithms (e.g., finding *all* mixed-strategy equilibria)
- Together, you get the best of both worlds: SageMath's mathematical power and Gambit's game-theoretic computation

This tutorial demonstrates:
1. Loading games from SageMath's built-in catalog and solving them with Gambit
2. Comparing equilibrium algorithms between SageMath and Gambit
3. Creating games in Gambit and bringing them into SageMath for further analysis
4. Using Gambit's exact rational solver to get precise equilibrium values

**Prerequisites:** This tutorial assumes you have read the PyGambit new user tutorials (01–04) and have a basic familiarity with Nash equilibria.

## Setup

You will need both `pygambit` and `SageMath` installed.

Install pygambit via pip:
```bash
pip install pygambit
```

For SageMath, see the [SageMath installation guide](https://doc.sagemath.org/html/en/installation/). You can also run this tutorial in a Jupyter notebook launched from within SageMath using `sage -n jupyter`.

> **Note:** When running inside SageMath's Jupyter kernel, integer literals are automatically converted to SageMath `Integer` objects. If you pass these directly to pygambit, you may need to wrap them with `int()`. For example: `game[int(0), int(0)][int(0)] = int(2)`. This tutorial uses `numpy` arrays to avoid this issue cleanly.

In [ ]:
import numpy as np
import pygambit as gbt

# SageMath imports — run inside a SageMath Jupyter kernel
from sage.game_theory.normal_form_game import NormalFormGame
import sage.game_theory.catalog_normal_form_games as catalog
from sage.matrix.constructor import matrix

## SageMath's Game Catalog

SageMath comes with a built-in catalog of classic normal-form games. Let's start with the **Prisoner's Dilemma**, one of the most well-known games in game theory.

In the Prisoner's Dilemma, two players each choose to either **Cooperate** or **Defect**. If both cooperate, they both get a moderate reward. But each player is individually tempted to defect, since defecting gives a better personal outcome regardless of the other player's choice.

In [ ]:
# Load Prisoner's Dilemma from SageMath's catalog
g_sage = catalog.PrisonersDilemma()
g_sage

In [ ]:
# Get the payoff matrices for both players
A, B = g_sage.payoff_matrices()
print("Player 1 payoff matrix (A):")
print(A)
print()
print("Player 2 payoff matrix (B):")
print(B)

## Loading a SageMath Game into Gambit

We can convert the SageMath payoff matrices into a pygambit `Game` object using `Game.from_arrays()`. This takes numpy arrays, so we convert the SageMath matrices first.

In [ ]:
# Convert SageMath matrices to numpy arrays
A_np = np.array(A.numpy(), dtype=float)
B_np = np.array(B.numpy(), dtype=float)

# Create a pygambit game from the arrays
g_gbt = gbt.Game.from_arrays(A_np, B_np, title="Prisoner's Dilemma")

# Add labels for the strategies
for player in g_gbt.players:
    player.strategies[0].label = "Cooperate"
    player.strategies[1].label = "Defect"

g_gbt

## Computing Nash Equilibria with Gambit's LCP Solver

Now let's compute the Nash equilibrium using Gambit's LCP (Linear Complementarity Programming) solver. The Prisoner's Dilemma has one unique Nash equilibrium: both players defect.

In [ ]:
# Compute Nash equilibria using Gambit's LCP solver
result = gbt.nash.lcp_solve(g_gbt, rational=False)

print(f"Algorithm used: {result.method}")
print(f"Number of equilibria found: {len(result.equilibria)}")
print()
for i, eq in enumerate(result.equilibria):
    print(f"Equilibrium {i+1}:")
    for player in g_gbt.players:
        probs = {s.label: round(float(eq[s]), 4) for s in player.strategies}
        print(f"  Player {player.number + 1}: {probs}")

As expected, both players choose to Defect with probability 1.0 — this is the dominant strategy equilibrium of the Prisoner's Dilemma.

## Comparing with SageMath's Built-in Solver

SageMath also has its own Nash equilibrium algorithms. Let's compare the results:

In [ ]:
# SageMath's enumeration algorithm
sage_result = g_sage.obtain_nash(algorithm='enumeration')
print("SageMath (enumeration):", sage_result)
print()

# Gambit's LCP solver
gbt_result = gbt.nash.lcp_solve(g_gbt, rational=False)
print("Gambit (LCP):")
for eq in gbt_result.equilibria:
    print(" ", eq)

## A Game with Multiple Equilibria: Battle of the Sexes

Now let's try a more interesting game — **Battle of the Sexes**. In this game, two players (Amy and Bob) want to spend the evening together, but they have different preferences. Amy prefers going to a Video Game event, Bob prefers a Movie. They get the highest reward when they are together, but their preferences differ on *which* event to attend.

This game has **three Nash equilibria** — two pure-strategy and one mixed-strategy equilibrium.

In [ ]:
# Load Battle of the Sexes from SageMath catalog
bos_sage = catalog.BattleOfTheSexes()
A_bos, B_bos = bos_sage.payoff_matrices()

print("Battle of the Sexes")
print("Player 1 (Amy) payoff matrix:")
print(A_bos)
print()
print("Player 2 (Bob) payoff matrix:")
print(B_bos)

In [ ]:
# Convert to pygambit
bos_gbt = gbt.Game.from_arrays(
    np.array(A_bos.numpy(), dtype=float),
    np.array(B_bos.numpy(), dtype=float),
    title="Battle of the Sexes"
)

# Label strategies
bos_gbt.players[0].label = "Amy"
bos_gbt.players[1].label = "Bob"
for player in bos_gbt.players:
    player.strategies[0].label = "Video Games"
    player.strategies[1].label = "Movie"

bos_gbt

## Finding ALL Equilibria with `enummixed_solve`

SageMath's LCP solver may not always find all equilibria for degenerate games. Gambit's `enummixed_solve` is designed to enumerate **all** mixed-strategy Nash equilibria.

We can also use `rational=True` to get **exact fractions** instead of floating-point approximations — this is one of Gambit's most powerful features.

In [ ]:
# Find ALL equilibria using enummixed_solve with exact rational arithmetic
all_eq = gbt.nash.enummixed_solve(bos_gbt, rational=True)

print(f"Total equilibria found: {len(all_eq.equilibria)}")
print()

for i, eq in enumerate(all_eq.equilibria):
    print(f"Equilibrium {i+1} (exact rational values):")
    print(f"  Amy:  Video Games = {eq['Amy']['Video Games']}, Movie = {eq['Amy']['Movie']}")
    print(f"  Bob:  Video Games = {eq['Bob']['Video Games']}, Movie = {eq['Bob']['Movie']}")
    print()

In [ ]:
# Compare with SageMath's enumeration — it finds the same equilibria
sage_eq = bos_sage.obtain_nash(algorithm='enumeration')
print("SageMath (enumeration) finds:")
for eq in sage_eq:
    print(" ", eq)

print()
print("Gambit (enummixed) finds:")
for eq in all_eq.equilibria:
    amy = [eq['Amy']['Video Games'], eq['Amy']['Movie']]
    bob = [eq['Bob']['Video Games'], eq['Bob']['Movie']]
    print(f"  Amy={amy}, Bob={bob}")

Notice that Gambit returns the mixed-strategy equilibrium values as **exact fractions** (e.g., `3/4` and `1/4`) while SageMath's enumeration also returns exact rational values.

For this game, both solvers agree. This is one of Gambit's key strengths: the ability to certify exact equilibria using rational arithmetic.

## Going the Other Direction: Gambit → SageMath

We can also create a game in pygambit and bring it back into SageMath for further mathematical analysis — such as computing payoff utilities symbolically or plotting best-response functions.

In [ ]:
# Create a game in pygambit: Matching Pennies
gbt_mp = gbt.Game.from_arrays(
    np.array([[ 1, -1], [-1,  1]], dtype=float),
    np.array([[-1,  1], [ 1, -1]], dtype=float),
    title="Matching Pennies"
)

for player in gbt_mp.players:
    player.strategies[0].label = "Heads"
    player.strategies[1].label = "Tails"

# Compute equilibrium in Gambit
mp_eq = gbt.nash.lcp_solve(gbt_mp, rational=True)
print("Equilibrium (from Gambit):")
print(mp_eq.equilibria[0])

In [ ]:
# Extract payoff arrays from the pygambit game
p1_arr, p2_arr = gbt_mp.to_arrays(dtype=float)

# Convert to SageMath matrices
A_sage = matrix(p1_arr.tolist())
B_sage = matrix(p2_arr.tolist())

# Create a SageMath NormalFormGame
sage_mp = NormalFormGame([A_sage, B_sage])
print("SageMath NormalFormGame:")
print(sage_mp)
print()

# Solve using SageMath's solver
print("Nash equilibria (from SageMath):")
print(sage_mp.obtain_nash(algorithm='enumeration'))

Both Gambit and SageMath correctly find the unique Nash equilibrium of Matching Pennies: both players mix equally between Heads (probability 1/2) and Tails (probability 1/2).

## Unique Strengths: SageMath's Exact Symbolic Math

One of SageMath's unique strengths is its **symbolic computation** engine. We can define a parameterized game where payoffs are symbolic expressions, and then solve for the equilibrium as a function of a parameter.

For example, let's see how the equilibrium strategy changes in a Coordination game as we change the relative payoffs:

In [ ]:
# Parameterized game in SageMath using symbolic variable
# Player 1's payoff matrix with parameter a
a = var('a')  # symbolic variable in SageMath

# A game where the payoff for (Cooperate, Cooperate) varies
A_sym = matrix([[a, 0], [0, 2]])
B_sym = matrix([[a, 0], [0, 2]])

print("Parameterized payoff matrix A (with symbolic variable 'a'):")
print(A_sym)
print()

# Evaluate for specific values
for val in [1, 2, 3, 4]:
    A_concrete = A_sym.subs(a=val)
    B_concrete = B_sym.subs(a=val)
    g_param = NormalFormGame([A_concrete, B_concrete])
    eqs = g_param.obtain_nash(algorithm='enumeration')
    print(f"  a={val}: Nash equilibria = {eqs}")

## Unique Strengths: Gambit's Algorithm Range

Gambit provides several algorithms that are **not available in SageMath**, including:

- `enumpure_solve` — finds all **pure-strategy** Nash equilibria
- `enummixed_solve` — finds all **mixed-strategy** Nash equilibria  
- `lcp_solve(rational=True)` — exact rational LCP computation
- `lp_solve` — for constant-sum games

Let's demonstrate `enumpure_solve` to find pure-strategy equilibria only:

In [ ]:
# Pure strategy Nash equilibria using Gambit
pure_eq = gbt.nash.enumpure_solve(bos_gbt)

print(f"Pure-strategy Nash equilibria in Battle of the Sexes: {len(pure_eq.equilibria)}")
print()
for i, eq in enumerate(pure_eq.equilibria):
    print(f"Pure Equilibrium {i+1}:")
    print(f"  Amy:  Video Games = {eq['Amy']['Video Games']}, Movie = {eq['Amy']['Movie']}")
    print(f"  Bob:  Video Games = {eq['Bob']['Video Games']}, Movie = {eq['Bob']['Movie']}")

Gambit correctly identifies the two pure-strategy Nash equilibria:
1. Both choose **Video Games** (Amy's preferred outcome)
2. Both choose **Movie** (Bob's preferred outcome)

## Summary

In this tutorial, we demonstrated how to use Gambit and SageMath together for game-theoretic analysis:

| Task | Tool Used |
|---|---|
| Load classic games quickly | SageMath catalog (`catalog.PrisonersDilemma()`) |
| Convert game to pygambit | `gbt.Game.from_arrays(np.array(A), np.array(B))` |
| Find one Nash equilibrium | `gbt.nash.lcp_solve(game)` |
| Find ALL Nash equilibria | `gbt.nash.enummixed_solve(game)` |
| Get exact rational results | `gbt.nash.lcp_solve(game, rational=True)` |
| Bring back to SageMath | `game.to_arrays()` → `NormalFormGame([A, B])` |
| Symbolic/parameterized games | SageMath `var()` and `matrix()` |

**Key takeaways:**
- SageMath's catalog makes it easy to load standard games without building them from scratch
- Gambit's `enummixed_solve` finds **all** mixed-strategy Nash equilibria reliably
- `rational=True` gives exact results — no floating-point approximation errors
- The two tools complement each other naturally: SageMath for symbolic math, Gambit for equilibrium computation

## What's Next?

- Explore more games from SageMath's catalog: see the [SageMath Game Theory Catalog reference](https://doc.sagemath.org/html/en/reference/game_theory/sage/game_theory/catalog_normal_form_games.html)
- Learn about Gambit's other algorithms in the [pygambit API documentation](https://gambitproject.readthedocs.io/en/latest/pygambit.api.html)
- See the other Gambit tutorials for more on building extensive-form games and computing equilibria in larger games